# Student Academic Performance — EDA & GPA Prediction

**10,000 students · 25 features · study habits, demographics, and GPA outcomes**

---

> **TL;DR** — This notebook provides a comprehensive exploration of the Student Academic Performance dataset. We examine how study hours, attendance, sleep, stress, tutoring, and socioeconomic factors shape GPA and pass/fail outcomes. We then build a **GPA regression model** (XGBoost, 5-fold CV R²) and a **pass/fail classifier**, and close with a **fairness analysis** across demographic groups.

**Contents:**
1. [Setup & Data Loading](#1)
2. [Dataset Overview](#2)
3. [Score Distributions](#3)
4. [Demographic Analysis](#4)
5. [Family Income Effect](#5)
6. [Study Hours Impact](#6)
7. [Sleep Analysis](#7)
8. [Attendance vs GPA](#8)
9. [Correlation Heatmap](#9)
10. [Feature Importance Baseline](#10)
11. [GPA Prediction (XGBoost Regression)](#11)
12. [Pass/Fail Classification](#12)
13. [Fairness Analysis](#13)
14. [ML Project Ideas](#14)
15. [Conclusion](#15)

---

If you find this exploration useful, please **upvote the dataset and this notebook**!

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

# Kaggle path fallback
KAGGLE_PATH = '/kaggle/input/student-academic-performance-dataset'
LOCAL_PATH  = '.'
base = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

df = pd.read_csv(f'{base}/students.csv')
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ── 2. Dataset Overview ───────────────────────────────────────────────────────
print('=== Shape ===')
print(df.shape)

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values.')

print('\n=== First 5 Rows ===')
df.head()

In [ ]:
# Numeric summary
df.describe(include='number').round(2)

In [ ]:
# ── 3. Score Distributions ────────────────────────────────────────────────────
score_cols = ['reading_score', 'writing_score', 'math_score', 'science_score', 'overall_gpa']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for ax, col, color in zip(axes, score_cols, colors):
    if col == 'overall_gpa':
        ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
        ax.set_xlabel('GPA (0-4)')
        ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean: {df[col].mean():.2f}')
    else:
        ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
        ax.set_xlabel('Score (0-100)')
        ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Academic Scores', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Pass rate:', f"{df['passed'].mean():.1%}")
print('Mean GPA :', f"{df['overall_gpa'].mean():.2f}")

In [ ]:
# ── 4. Demographic Analysis ───────────────────────────────────────────────────
subject_scores = ['reading_score', 'writing_score', 'math_score', 'science_score']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender
gender_means = df.groupby('gender')[subject_scores].mean()
gender_means.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Mean Subject Scores by Gender')
axes[0].set_ylabel('Mean Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(loc='lower right', fontsize=8)

# Ethnicity
eth_means = df.groupby('ethnicity')[subject_scores].mean()
eth_means.plot(kind='bar', ax=axes[1], colormap='Set1', edgecolor='white')
axes[1].set_title('Mean Subject Scores by Ethnicity')
axes[1].set_ylabel('Mean Score')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(loc='lower right', fontsize=8)

# Parental education
edu_order = ['none', 'high_school', 'some_college', 'bachelor', 'master', 'phd']
edu_means = df.groupby('parental_education')['overall_gpa'].mean().reindex(edu_order)
bars = axes[2].bar(range(len(edu_means)), edu_means.values,
                   color=plt.cm.Blues(np.linspace(0.3, 0.9, len(edu_means))),
                   edgecolor='white')
axes[2].set_xticks(range(len(edu_means)))
axes[2].set_xticklabels([e.replace('_', '\n') for e in edu_order], fontsize=8)
axes[2].set_title('Mean GPA by Parental Education')
axes[2].set_ylabel('Mean GPA')
for bar, val in zip(bars, edu_means.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Family Income Effect ───────────────────────────────────────────────────
income_order = ['low', 'middle', 'high']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot GPA by income
income_groups = [df[df['family_income'] == inc]['overall_gpa'].values for inc in income_order]
bp = axes[0].boxplot(income_groups, labels=income_order, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
colors_box = ['#e74c3c', '#f39c12', '#2ecc71']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('GPA Distribution by Family Income')
axes[0].set_ylabel('Overall GPA')
axes[0].set_xlabel('Family Income Bracket')

# Resource access by income
resource_df = df.groupby('family_income').agg(
    internet_pct=('internet_access', lambda x: (x == 'yes').mean() * 100),
    laptop_pct=('has_laptop', lambda x: (x == 'yes').mean() * 100),
).reindex(income_order)
x = np.arange(len(income_order))
width = 0.35
axes[1].bar(x - width/2, resource_df['internet_pct'], width, label='Internet Access',
            color='#3498db', alpha=0.8, edgecolor='white')
axes[1].bar(x + width/2, resource_df['laptop_pct'], width, label='Has Laptop',
            color='#9b59b6', alpha=0.8, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(income_order)
axes[1].set_title('Resource Access by Family Income')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Mean GPA by income:')
print(df.groupby('family_income')['overall_gpa'].mean().reindex(income_order).round(3))

In [ ]:
# ── 6. Study Hours Impact ─────────────────────────────────────────────────────
from scipy.stats import pearsonr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: study hours vs GPA
sample = df.sample(2000, random_state=42)
axes[0].scatter(sample['study_hours_per_week'], sample['overall_gpa'],
                alpha=0.25, s=15, color='steelblue')

# OLS trend line
m, b = np.polyfit(df['study_hours_per_week'], df['overall_gpa'], 1)
x_line = np.linspace(0, 40, 100)
axes[0].plot(x_line, m * x_line + b, color='red', linewidth=2,
             label=f'Trend (slope={m:.3f})')
r, p = pearsonr(df['study_hours_per_week'], df['overall_gpa'])
axes[0].set_title(f'Study Hours vs GPA  (r={r:.2f}, p<0.001)')
axes[0].set_xlabel('Study Hours Per Week')
axes[0].set_ylabel('Overall GPA')
axes[0].legend()

# Mean GPA binned by study hours
df['study_bin'] = pd.cut(df['study_hours_per_week'],
                          bins=[0, 5, 10, 15, 20, 25, 30, 40],
                          labels=['0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30+'])
bin_gpa = df.groupby('study_bin', observed=True)['overall_gpa'].mean()
bin_counts = df.groupby('study_bin', observed=True)['overall_gpa'].count()

bars = axes[1].bar(range(len(bin_gpa)), bin_gpa.values,
                   color=plt.cm.Blues(np.linspace(0.3, 0.9, len(bin_gpa))),
                   edgecolor='white')
axes[1].set_xticks(range(len(bin_gpa)))
axes[1].set_xticklabels(bin_gpa.index, rotation=30)
axes[1].set_title('Mean GPA by Study Hours Band')
axes[1].set_xlabel('Weekly Study Hours')
axes[1].set_ylabel('Mean GPA')
for bar, val, n in zip(bars, bin_gpa.values, bin_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{val:.2f}\n(n={n})', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()
df.drop(columns=['study_bin'], inplace=True)

In [ ]:
# ── 7. Sleep Analysis (Quadratic Effect) ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + quadratic fit
sample = df.sample(2000, random_state=7)
axes[0].scatter(sample['sleep_hours'], sample['overall_gpa'],
                alpha=0.25, s=15, color='#9b59b6')

# Quadratic fit
coeffs = np.polyfit(df['sleep_hours'], df['overall_gpa'], 2)
x_fit = np.linspace(4, 10, 200)
y_fit = np.polyval(coeffs, x_fit)
axes[0].plot(x_fit, y_fit, color='red', linewidth=2, label='Quadratic fit')
optimal = -coeffs[1] / (2 * coeffs[0])
axes[0].axvline(optimal, color='orange', linestyle='--', linewidth=1.5,
                label=f'Optimal: {optimal:.1f}h')
axes[0].set_title('Sleep Hours vs GPA (quadratic)')
axes[0].set_xlabel('Sleep Hours Per Night')
axes[0].set_ylabel('Overall GPA')
axes[0].legend()

# Mean GPA by sleep bucket
df['sleep_bin'] = pd.cut(df['sleep_hours'],
                          bins=[3.9, 5, 6, 7, 8, 9, 10.1],
                          labels=['4-5', '5-6', '6-7', '7-8', '8-9', '9-10'])
sleep_gpa = df.groupby('sleep_bin', observed=True)['overall_gpa'].mean()

peak_idx = sleep_gpa.values.argmax()
bar_colors = ['#e74c3c' if i != peak_idx else '#2ecc71'
              for i in range(len(sleep_gpa))]
axes[1].bar(range(len(sleep_gpa)), sleep_gpa.values, color=bar_colors, edgecolor='white')
axes[1].set_xticks(range(len(sleep_gpa)))
axes[1].set_xticklabels(sleep_gpa.index)
axes[1].set_title('Mean GPA by Sleep Hours Band')
axes[1].set_xlabel('Sleep Hours Per Night')
axes[1].set_ylabel('Mean GPA')
axes[1].set_ylim(sleep_gpa.min() * 0.95, sleep_gpa.max() * 1.05)
axes[1].text(peak_idx, sleep_gpa.values[peak_idx] + 0.003,
             'Optimal', ha='center', fontsize=9, color='darkgreen')

plt.tight_layout()
plt.show()
df.drop(columns=['sleep_bin'], inplace=True)
print(f'Estimated optimal sleep: {optimal:.1f} hours/night')

In [ ]:
# ── 8. Attendance vs GPA ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
sample = df.sample(2000, random_state=8)
axes[0].scatter(sample['attendance_rate'], sample['overall_gpa'],
                alpha=0.25, s=15, color='#27ae60')
m2, b2 = np.polyfit(df['attendance_rate'], df['overall_gpa'], 1)
x2 = np.linspace(50, 100, 100)
axes[0].plot(x2, m2 * x2 + b2, color='red', linewidth=2)
r2, _ = pearsonr(df['attendance_rate'], df['overall_gpa'])
axes[0].set_title(f'Attendance Rate vs GPA  (r={r2:.2f})')
axes[0].set_xlabel('Attendance Rate (%)')
axes[0].set_ylabel('Overall GPA')

# Pass rate by attendance tier
df['att_tier'] = pd.cut(df['attendance_rate'],
                         bins=[49, 65, 75, 85, 95, 101],
                         labels=['50-65%', '65-75%', '75-85%', '85-95%', '95-100%'])
tier_pass = df.groupby('att_tier', observed=True)['passed'].mean() * 100
axes[1].bar(range(len(tier_pass)), tier_pass.values,
            color=plt.cm.Greens(np.linspace(0.3, 0.9, len(tier_pass))), edgecolor='white')
axes[1].set_xticks(range(len(tier_pass)))
axes[1].set_xticklabels(tier_pass.index, rotation=15)
axes[1].set_title('Pass Rate by Attendance Tier')
axes[1].set_xlabel('Attendance Rate')
axes[1].set_ylabel('Pass Rate (%)')
for i, v in enumerate(tier_pass.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
df.drop(columns=['att_tier'], inplace=True)

In [ ]:
# ── 9. Correlation Heatmap ─────────────────────────────────────────────────────
# Encode categoricals for correlation
corr_df = df.copy()

ordinal_maps = {
    'parental_education':  {'none': 0, 'high_school': 1, 'some_college': 2,
                             'bachelor': 3, 'master': 4, 'phd': 5},
    'family_income':       {'low': 0, 'middle': 1, 'high': 2},
    'parental_involvement':{'low': 0, 'medium': 1, 'high': 2},
    'internet_access':     {'no': 0, 'yes': 1},
    'has_laptop':          {'no': 0, 'yes': 1},
    'sports_participation':{'no': 0, 'yes': 1},
}
for col, mapping in ordinal_maps.items():
    corr_df[col] = corr_df[col].map(mapping)

numeric_cols = [
    'study_hours_per_week', 'attendance_rate', 'tutoring_sessions',
    'sleep_hours', 'stress_level', 'motivation_score',
    'parental_education', 'family_income', 'parental_involvement',
    'internet_access', 'has_laptop', 'extracurricular_activities',
    'reading_score', 'writing_score', 'math_score', 'science_score',
    'overall_gpa', 'passed',
]

corr_matrix = corr_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 7},
)
ax.set_title('Correlation Heatmap — All Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with GPA
gpa_corr = corr_matrix['overall_gpa'].drop('overall_gpa').sort_values(key=abs, ascending=False)
print('Top correlations with overall_gpa:')
print(gpa_corr.round(3).head(10).to_string())

In [ ]:
# ── 10. Feature Importance Baseline (Mutual Information) ──────────────────────
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder

mi_df = df.copy()

# Encode all categoricals
cat_cols = mi_df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'student_id']
le = LabelEncoder()
for col in cat_cols:
    mi_df[col] = le.fit_transform(mi_df[col].astype(str))

feature_cols = [
    c for c in mi_df.columns
    if c not in ('student_id', 'overall_gpa', 'passed',
                 'reading_score', 'writing_score', 'math_score', 'science_score')
]

X_mi = mi_df[feature_cols]
y_mi = mi_df['overall_gpa']

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_mi = ['#e74c3c' if v == mi_series.max() else '#3498db' for v in mi_series.values]
ax.barh(range(len(mi_series)), mi_series.values, color=colors_mi, edgecolor='white')
ax.set_yticks(range(len(mi_series)))
ax.set_yticklabels(mi_series.index)
ax.set_title('Mutual Information Scores vs overall_gpa\n(higher = more predictive)', fontweight='bold')
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print('Top 5 features by mutual information:')
print(mi_series.sort_values(ascending=False).head(5).round(4).to_string())

In [ ]:
# ── 11. GPA Prediction — XGBoost Regression ───────────────────────────────────
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

model_df = df.copy()

cat_cols = model_df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'student_id']
le = LabelEncoder()
for col in cat_cols:
    model_df[col] = le.fit_transform(model_df[col].astype(str))

reg_features = [
    c for c in model_df.columns
    if c not in ('student_id', 'overall_gpa', 'passed',
                 'reading_score', 'writing_score', 'math_score', 'science_score')
]

X_reg = model_df[reg_features]
y_reg = model_df['overall_gpa']

xgb_reg = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = np.sqrt(-cross_val_score(xgb_reg, X_reg, y_reg, cv=cv,
                                        scoring='neg_mean_squared_error'))
r2_scores   = cross_val_score(xgb_reg, X_reg, y_reg, cv=cv, scoring='r2')

print('=== GPA Regression — 5-Fold CV Results ===')
print(f'  RMSE : {rmse_scores.mean():.4f} (+/- {rmse_scores.std():.4f})')
print(f'  R²   : {r2_scores.mean():.4f} (+/- {r2_scores.std():.4f})')

# Feature importance
xgb_reg.fit(X_reg, y_reg)
feat_imp = pd.Series(xgb_reg.feature_importances_,
                     index=reg_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
top_feats = feat_imp.tail(15)
top_colors = ['#e74c3c' if v == top_feats.max() else '#3498db' for v in top_feats.values]
ax.barh(range(len(top_feats)), top_feats.values, color=top_colors, edgecolor='white')
ax.set_yticks(range(len(top_feats)))
ax.set_yticklabels(top_feats.index)
ax.set_title('XGBoost Feature Importance (Top 15) — GPA Prediction', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── 12. Pass/Fail Classification ─────────────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X_clf = model_df[reg_features]
y_clf = model_df['passed']

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_scores = cross_val_score(xgb_clf, X_clf, y_clf, cv=skf, scoring='roc_auc')
y_pred = cross_val_predict(xgb_clf, X_clf, y_clf, cv=skf)

print(f'5-Fold CV ROC-AUC: {roc_scores.mean():.4f} (+/- {roc_scores.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_clf, y_pred, target_names=['Fail', 'Pass']))

# Confusion matrix
cm = confusion_matrix(y_clf, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Fail', 'Pred Pass'],
            yticklabels=['True Fail', 'True Pass'])
ax.set_title('Confusion Matrix — Pass/Fail Classification', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Fairness Analysis ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean GPA and pass rate by ethnicity
eth_stats = df.groupby('ethnicity').agg(
    mean_gpa=('overall_gpa', 'mean'),
    pass_rate=('passed', 'mean'),
    count=('student_id', 'count'),
).sort_values('mean_gpa', ascending=False)

axes[0].bar(eth_stats.index, eth_stats['mean_gpa'],
            color=plt.cm.Set2(range(len(eth_stats))), edgecolor='white')
axes[0].axhline(df['overall_gpa'].mean(), color='red', linestyle='--', linewidth=1.5,
                label='Overall mean')
axes[0].set_title('Mean GPA by Ethnicity')
axes[0].set_ylabel('Mean GPA')
axes[0].legend()

# Pass rate by gender
gender_stats = df.groupby('gender').agg(
    pass_rate=('passed', 'mean'),
    mean_gpa=('overall_gpa', 'mean'),
)
axes[1].bar(gender_stats.index, gender_stats['pass_rate'] * 100,
            color=['#3498db', '#e74c3c', '#9b59b6'], edgecolor='white', alpha=0.85)
axes[1].set_title('Pass Rate by Gender')
axes[1].set_ylabel('Pass Rate (%)')
for i, (idx, row) in enumerate(gender_stats.iterrows()):
    axes[1].text(i, row['pass_rate'] * 100 + 0.3,
                 f"{row['pass_rate']*100:.1f}%", ha='center', fontsize=10)

# GPA gap: private vs public vs charter
school_stats = df.groupby('school_type').agg(
    mean_gpa=('overall_gpa', 'mean'),
    pass_rate=('passed', 'mean'),
    count=('student_id', 'count'),
).sort_values('mean_gpa', ascending=False)
axes[2].bar(school_stats.index, school_stats['mean_gpa'],
            color=['#2ecc71', '#3498db', '#f39c12'], edgecolor='white', alpha=0.85)
axes[2].axhline(df['overall_gpa'].mean(), color='red', linestyle='--', linewidth=1.5,
                label='Overall mean')
axes[2].set_title('Mean GPA by School Type')
axes[2].set_ylabel('Mean GPA')
axes[2].legend()

plt.suptitle('Fairness Analysis — Performance Gaps Across Groups',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Ethnicity breakdown:')
print(eth_stats.round(3).to_string())
print()
print('School type breakdown:')
print(school_stats.round(3).to_string())

## ML Project Ideas

| # | Project | Complexity | What You Learn |
|---|---------|------------|----------------|
| 1 | GPA Regression (XGBoost / LightGBM) | Beginner | Regression, feature engineering, CV |
| 2 | Pass/Fail Classification | Beginner | Binary classification, precision/recall, ROC-AUC |
| 3 | Feature Importance & SHAP Explanations | Intermediate | Explainability, SHAP values, partial dependence plots |
| 4 | Multi-Output Regression (predict all 4 subject scores simultaneously) | Intermediate | Multi-output models, task correlation |
| 5 | Fairness-Aware Modeling (minimize performance gap across demographic groups) | Intermediate | Algorithmic fairness, group metrics, reweighting |
| 6 | Student Clustering / Profiling (K-Means or UMAP) | Intermediate | Unsupervised learning, dimensionality reduction |
| 7 | Early Intervention Model (identify at-risk students from mid-term features) | Advanced | Imbalanced classification, cost-sensitive learning |
| 8 | Causal Inference (does tutoring *cause* better grades, controlling for confounders?) | Advanced | DoWhy, propensity score matching, counterfactuals |

## Conclusion & Next Steps

### Key Findings

1. **Study hours** is the single strongest predictor of GPA — each additional hour/week adds roughly 0.05-0.07 GPA points on average.
2. **Attendance rate** is the second most powerful factor: students attending 95-100% of classes pass at dramatically higher rates.
3. **Sleep** shows a clear quadratic relationship with performance — the optimal is approximately 7.5 hours/night; both under- and over-sleeping hurt.
4. **Stress** is a meaningful negative predictor; **motivation** is a meaningful positive predictor — psychological factors matter beyond study time alone.
5. **Family income** and **parental education** each add modest but consistent boosts, driven in part by resource access (laptop, internet) and parental involvement.
6. **Private schools** show a small GPA premium over public schools; the gap is modest (~0.1 GPA points) once study habits are controlled.
7. **XGBoost regression** achieves strong predictive performance, confirming the dataset contains genuine signal for GPA prediction tasks.
8. The **fairness analysis** reveals moderate GPA gaps across ethnicity and school type groups — rich ground for equity-focused ML research.

### Next Steps

1. Try SHAP values to explain individual student predictions and compare with the mutual information baseline.
2. Build a multi-output model that predicts all four subject scores simultaneously.
3. Explore causal inference: does attending tutoring sessions **cause** better grades, or do already-motivated students self-select into tutoring?

---

**Dataset by Lorenzo Scaturchio.**

### **If you found this notebook useful, please upvote both the dataset and this notebook! It helps the community discover quality educational resources.**